##### Import Libraries

In [3]:
import os
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam, SGD

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,confusion_matrix,classification_report,ConfusionMatrixDisplay)

In [4]:
print("Tensorflow version: ", tf.__version__)
print("Num GPUs Available: ",len(tf.config.list_physical_devices('GPU')))

Tensorflow version:  2.21.0
Num GPUs Available:  0


In [6]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("Random seed set to: ",SEED)

Random seed set to:  42


In [9]:
DATASET_DIR=Path("RealWaste")
class_names=sorted([folder.name for folder in DATASET_DIR.iterdir() if folder.is_dir()])
NUM_CLASSES=len(class_names)
print("No. of classes: ",len(class_names))
print("Classes: ")
for i, class_name in enumerate(class_names):
    print(class_name)

No. of classes:  9
Classes: 
Cardboard
Food Organics
Glass
Metal
Miscellaneous Trash
Paper
Plastic
Textile Trash
Vegetation


In [16]:
class_counts = {}

for class_name in class_names:
    class_dir=DATASET_DIR/class_name
    count=sum(1 for file in class_dir.iterdir())
    class_counts[class_name]=count
class_counts_df=pd.DataFrame(list(class_counts.items()),columns=["Class","Number of Images"])
class_counts_df

,Class,Number of Images
0,Cardboard,461
1,Food Organics,411
2,Glass,420
3,Metal,790
4,Miscellaneous Trash,495
5,Paper,500
6,Plastic,921
7,Textile Trash,318
8,Vegetation,436


In [22]:
TOTAL_IMAGES=class_counts_df["Number of Images"].sum()

##### Data Preprocessing

In [25]:
image_paths=[]
labels=[]
for label, class_name in enumerate(class_names):
    class_dir=DATASET_DIR/class_name
    for file in class_dir.iterdir():
        image_paths.append(str(file))
        labels.append(label)

image_paths=np.array(image_paths)
labels=np.array(labels)

print("Total images:",len(image_paths))


Total images: 4752


In [27]:
# data spliting
X_train, X_temp, y_train, y_temp = train_test_split(image_paths,labels,test_size=0.30,stratify=labels,random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp,y_temp,test_size=0.50,stratify=y_temp,random_state=SEED)

print("Training images:", len(X_train))
print("Validation images:", len(X_val))
print("Testing images:", len(X_test))

Training images: 3326
Validation images: 713
Testing images: 713


In [28]:
# Tensorflow input
IMG_SIZE=(64,64)
BATCH_SIZE=32

def load_image(path,label):
    image=tf.io.read_file(path)
    image=tf.image.decode_image(image,channels=3,expand_animations=False)
    image=tf.image.resize(image,IMG_SIZE)
    image=tf.cast(image,tf.float32)/255.0
    return image,label

In [30]:
train_ds=tf.data.Dataset.from_tensor_slices((X_train,y_train))
val_ds=tf.data.Dataset.from_tensor_slices((X_val,y_val))
test_ds=tf.data.Dataset.from_tensor_slices((X_test,y_test))

train_ds = train_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

train_ds = train_ds.shuffle(len(X_train), seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

#### Model A - Standard CNN

In [39]:
# Build model A
model_A=models.Sequential([
    layers.Input(shape=(64,64,3)),

    layers.Conv2D(16,(3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(32,(3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64,(3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),

    layers.Dense(64, activation="relu"),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

In [40]:
model_A.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 64, 64, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 32, 32, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 32, 32, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │       262,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 9)              │           585 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 286,377 (1.09 MB)

 Trainable params: 286,377 (1.09 MB)

 Non-trainable params: 0 (0.00 B)

#### Model B - Lightweight CNN

In [45]:
# Build model B
model_B=models.Sequential([
    layers.Input(shape=(64,64,3)),

    layers.SeparableConv2D(16,(3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.SeparableConv2D(32,(3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.SeparableConv2D(64,(3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.GlobalAveragePooling2D(),

    layers.Dense(32, activation="relu"),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

In [46]:
model_B.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ separable_conv2d_6              │ (None, 64, 64, 16)     │            91 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_21 (MaxPooling2D) │ (None, 32, 32, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_7              │ (None, 32, 32, 32)     │           688 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_22 (MaxPooling2D) │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_8              │ (None, 16, 16, 64)     │         2,400 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_23 (MaxPooling2D) │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 9)              │           297 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,556 (21.70 KB)

 Trainable params: 5,556 (21.70 KB)

 Non-trainable params: 0 (0.00 B)